# Phase 4: Ensembling and Final Submission
This notebook combines the predictions from CatBoost and TFT (if available) and generates the final submission file.

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Load the original test dataframe to get the 'Index' column
full_df = pd.read_csv('../data/dataset/preprocessed_full_data.csv')
test_df = full_df[full_df['split'] == 'test'].copy()

# 2. Load CatBoost predictions (Baseline)
cat_preds = np.load('../data/dataset/cat_preds.npy')
print("Loaded CatBoost predictions.")

# 3. Try to load TFT predictions, fallback to CatBoost baseline if not found
tft_path = '../data/dataset/tft_preds.npy'
if os.path.exists(tft_path):
    print("TFT predictions found! Applying weighted ensemble (0.6 TFT + 0.4 CatBoost).")
    tft_preds = np.load(tft_path)
    final_preds = (0.6 * tft_preds) + (0.4 * cat_preds)
else:
    print("TFT predictions not found. Defaulting to 100% CatBoost baseline.")
    final_preds = cat_preds

# 4. Boundary Safety: The R2 metric punishes negative values severely
final_preds = np.clip(final_preds, a_min=0.0, a_max=None)

# 5. File Generation
submission_df = pd.DataFrame({
    'Index': test_df['Index'].astype(int),
    'demand': final_preds
})

# Save to CSV
submission_path = '../data/dataset/submission.csv'
submission_df.to_csv(submission_path, index=False)

# 6. Final Validation Assertions
assert submission_df.shape == (41778, 2), f"Shape mismatch! Expected (41778, 2), got {submission_df.shape}"
assert list(submission_df.columns) == ['Index', 'demand'], "Column names are incorrect!"
assert submission_df['demand'].min() >= 0, "Negative demands found!"

print(f"\n✅ Success! Final submission saved to: {submission_path}")
display(submission_df.head())